# Part 2: Simple Logistic Regression Model
We will be using a simple bag-of-words representation for our initial, simple model. To train this model, we will be using the pytorch library.

In [1]:
from transformers import *
from pathlib import Path
import numpy as np

DATA_DIR = Path("data/preprocessed")
TOKENIZER_DIR = DATA_DIR / "tokenizers"
SPLIT_DIR = DATA_DIR / "splits"

In [2]:
from sklearn.pipeline import Pipeline
from joblib import load


tokenizer : TokenTransformer = load(TOKENIZER_DIR / "top10k.joblib")
pipeline = Pipeline([
    ("tokenize", PrefitTransformer(tokenizer)), 
    ("vectorize", BagofwordsTransformer("content", tokenizer.size()))
])
pipeline

,steps,"[('tokenize', ...), ('vectorize', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformer,TokenTransfor... top_n=10000)
,top_n,10000
,special_tokens,"['<NUM>', '<DATE>', ...]"
,stopwords,"{'a', 'about', 'above', 'after', 'again', 'against', ...}"
,stem,True
,column,'content'
,vocab_size,10000


In [3]:
import pandas as pd
df_train = pd.read_csv(SPLIT_DIR / "train.csv")
df_train.head()

,id,domain,type,url,content,title,is_reliable
0,1033553.0,beforeitsnews.com,fake,http://beforeitsnews.com/media/2011/10/liberal...,liberal 'ethicists' line up angrily against cn...,liberal 'ethicists' line up angrily against cn...,0
1,3381445.0,dailykos.com,political,https://www.dailykos.com/stories/2014/10/13/13...,hey! good evening! this evening's music featu...,the evening blues,1
2,3989370.0,thinkprogress.org,political,https://thinkprogress.org/7-reasons-north-caro...,in the crowded north carolina primary race for...,<NUM> reasons north carolina’s ‘establishment ...,1
3,8627669.0,nytimes.com,reliable,https://www.nytimes.com/2003/02/09/arts/dance-...,"for the fourth act's opening scene, when aenea...","even divas, it seems, can dance (a little, at ...",1
4,4681740.0,dailykos.com,political,https://www.dailykos.com/stories/2016/08/24/15...,companies that break labor laws won’t be able ...,obama order will ban labor law violators from ...,1


In [4]:
df_val = pd.read_csv(SPLIT_DIR / "val.csv")
df_val.head()

,id,domain,type,url,content,title,is_reliable
0,4927322.0,pravda.ru,bias,https://www.pravda.ru/accidents/03-10-2010/105...,"столичные ""зебры"" опасны для детей минувшее во...","столичные ""зебры"" опасны для детей",1
1,7123143.0,americablog.com,bias,http://americablog.com/2009/01/obama-signs-ord...,why does barack obama love the constitution? t...,obama signs order to close guantanamo in a year,1
2,9836763.0,nytimes.com,reliable,https://www.nytimes.com/2017/04/24/us/politics...,"hoping to pad the report card, he announced su...",trump wants it known: grading <NUM> days is ‘r...,1
3,2422578.0,twitchy.com,clickbait,https://twitchy.com/jennqpublic-3135/2013/04/2...,like gov. rick perry and anyone with even a sm...,amen: ted cruz on vile sac bee cartoon: ‘liber...,1
4,2968491.0,conservapedia.com,bias,http://www.conservapedia.com/Special:Log/Aweso...,combined display of all available logs of cons...,all public logs,1


In [5]:
X_train, y_train = pipeline.fit_transform(df_train), df_train["is_reliable"].to_numpy()
X_val, y_val = pipeline.transform(df_val), df_val["is_reliable"].to_numpy()

100%|██████████| 79337/79337 [00:02<00:00, 30855.54it/s]
d:\Programming\GDS2026\.venv\lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [6]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    verbose=1)

print("fitting")
model.fit(X_train,y_train)

fitting


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:  2.6min finished


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [7]:
from sklearn.metrics import f1_score

y_pred = model.predict(X_val)
print("F1 score of logistic regression model: ", f1_score(y_val, y_pred))

F1 score of logistic regression model:  0.8872781427615394


## 2.1. Including content
As we saw in the data analysis section, the label of an article is strongly controlled by the domain it originates from, so including this information should surely improve the model

In [8]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


tokenizer : TokenTransformer = load(TOKENIZER_DIR / "top10k.joblib")
pipeline = Pipeline([
    ("transform", ColumnTransformer(
        transformers=[
            ("domain", OneHotEncoder(handle_unknown="ignore"), ["domain"]),
            ("content", Pipeline([
                ("tokenize", PrefitTransformer(tokenizer)), 
                ("vectorize", BagofwordsTransformer("content", tokenizer.size()))
            ]), ["content"])
        ]))
])
pipeline

,steps,"[('transform', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('domain', ...), ('content', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [9]:
X_train, y_train = pipeline.fit_transform(df_train), df_train["is_reliable"].to_numpy()
X_val, y_val = pipeline.transform(df_val), df_val["is_reliable"].to_numpy()

100%|██████████| 79337/79337 [00:00<00:00, 4407580.29it/s]
d:\Programming\GDS2026\.venv\lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [10]:
model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    verbose=1)

print("fitting")
model.fit(X_train,y_train)

fitting


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.2s finished


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [11]:
from sklearn.metrics import f1_score

y_pred = model.predict(X_val)
print("F1 score of logistic regression model including domain: ", f1_score(y_val, y_pred))

F1 score of logistic regression model including domain:  0.9996158278908951
